<a href="https://colab.research.google.com/github/Nikita-Puzyrev/ML-2026/blob/main/labwork2/labwork2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

DATA_URL = (
    "https://archive.ics.uci.edu/static/public/560/"
    "seoul%2Bbike%2Bsharing%2Bdemand.zip"
)

DATA_DIR = "data"
ZIP_PATH = os.path.join(DATA_DIR, "seoul_bike.zip")

WINDOW_SIZE = 24

TRAIN_PART = 0.70
VAL_PART = 0.15
TEST_PART = 0.15

LEARNING_RATE = 0.005
EPOCHS = 300
BATCH_SIZE = 128

HIDDEN_SIZES = [4, 8, 16]

RANDOM_SEED = 42


# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================================

def download_dataset():
    """
    Загружает набор данных с UCI и распаковывает его.
    """

    os.makedirs(DATA_DIR, exist_ok=True)

    csv_path = os.path.join(DATA_DIR, "SeoulBikeData.csv")

    # Если CSV уже существует, второй раз не скачиваем
    if os.path.exists(csv_path):
        print("Данные уже загружены.")
        return csv_path

    print("Скачивание набора данных...")

    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

    print("Распаковка архива...")

    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        archive.extractall(DATA_DIR)

    print("Данные успешно загружены.")

    return csv_path


# ============================================================
# 3. ЧТЕНИЕ CSV
# ============================================================

def load_dataframe(csv_path):
    """
    Загружает CSV.

    В исходном файле могут встречаться символы,
    из-за которых кодировка UTF-8 определяется некорректно,
    поэтому пробуем несколько вариантов.
    """

    encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1",
        "unicode_escape"
    ]

    for encoding in encodings:
        try:
            df = pd.read_csv(csv_path, encoding=encoding)

            print(f"Файл прочитан в кодировке: {encoding}")

            return df

        except UnicodeDecodeError:
            continue

    raise RuntimeError("Не удалось определить кодировку CSV.")


# ============================================================
# 4. ПОДГОТОВКА ВРЕМЕННОГО РЯДА
# ============================================================

def prepare_series(df):

    print("\nНазвания столбцов:")
    print(df.columns.tolist())

    target_column = "Rented Bike Count"

    if target_column not in df.columns:
        raise ValueError(
            f"Столбец '{target_column}' не найден."
        )

    # Создаем полноценную временную метку
    df["Timestamp"] = (
        pd.to_datetime(
            df["Date"],
            dayfirst=True,
            errors="coerce"
        )
        +
        pd.to_timedelta(
            df["Hour"],
            unit="h"
        )
    )

    # Сортируем данные по времени
    df = df.sort_values("Timestamp")

    # Удаляем строки, если дата по какой-либо причине
    # не смогла быть преобразована
    df = df.dropna(subset=["Timestamp"])

    values = df[target_column].astype(float).to_numpy()

    timestamps = df["Timestamp"].to_numpy()

    return values, timestamps


# ============================================================
# 5. ВИЗУАЛИЗАЦИЯ ИСХОДНОГО РЯДА
# ============================================================

def plot_original_series(values, timestamps):

    plt.figure(figsize=(14, 5))

    plt.plot(timestamps, values)

    plt.xlabel("Дата")
    plt.ylabel("Количество арендованных велосипедов")

    plt.title("Исходный временной ряд")

    plt.grid()

    plt.tight_layout()

    plt.show()


# ============================================================
# 6. НОРМАЛИЗАЦИЯ
# ============================================================

def normalize(data, minimum, maximum):

    return (
        (data - minimum)
        /
        (maximum - minimum)
    )


def denormalize(data, minimum, maximum):

    return (
        data * (maximum - minimum)
        + minimum
    )


# ============================================================
# 7. СОЗДАНИЕ ВРЕМЕННЫХ ОКОН
# ============================================================

def create_windows(data, window_size):
    """
    Пример:

    [10, 12, 15, 13, 16, 18]

    window_size = 3

    Получаем:

    [10, 12, 15] -> 13
    [12, 15, 13] -> 16
    [15, 13, 16] -> 18
    """

    X = []
    y = []

    for i in range(window_size, len(data)):

        # Предыдущие window_size значений
        window = data[i - window_size:i]

        # Следующее значение
        target = data[i]

        X.append(window)

        y.append(target)

    X = np.array(X)

    y = np.array(y).reshape(-1, 1)

    return X, y


# ============================================================
# 8. НЕЙРОННАЯ СЕТЬ
# ============================================================

class NeuralNetwork:

    def __init__(
        self,
        input_size,
        hidden_size,
        learning_rate=0.001,
        seed=42
    ):

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate

        rng = np.random.default_rng(seed)

        # ----------------------------------------------------
        # Веса первого слоя
        #
        # input_size -> hidden_size
        # ----------------------------------------------------

        self.W1 = rng.normal(
            0,
            np.sqrt(2 / input_size),
            size=(input_size, hidden_size)
        )

        self.b1 = np.zeros(
            (1, hidden_size)
        )

        # ----------------------------------------------------
        # Веса выходного слоя
        #
        # hidden_size -> 1
        # ----------------------------------------------------

        self.W2 = rng.normal(
            0,
            np.sqrt(2 / hidden_size),
            size=(hidden_size, 1)
        )

        self.b2 = np.zeros(
            (1, 1)
        )

        self.history = {
            "train_loss": [],
            "val_loss": []
        }


    # ========================================================
    # ReLU
    # ========================================================

    @staticmethod
    def relu(x):

        return np.maximum(0, x)


    # ========================================================
    # Производная ReLU
    # ========================================================

    @staticmethod
    def relu_derivative(x):

        return (x > 0).astype(float)


    # ========================================================
    # MSE
    # ========================================================

    @staticmethod
    def mse(y_true, y_pred):

        return np.mean(
            (y_true - y_pred) ** 2
        )


    # ========================================================
    # ПРЯМОЕ РАСПРОСТРАНЕНИЕ
    # ========================================================

    def forward(self, X):

        # ------------------------------
        # Скрытый слой
        # ------------------------------

        self.Z1 = (
            X @ self.W1
            +
            self.b1
        )

        self.A1 = self.relu(
            self.Z1
        )

        # ------------------------------
        # Выходной слой
        # ------------------------------

        # Здесь нет ReLU или sigmoid,
        # поскольку решается задача регрессии

        self.Y_pred = (
            self.A1 @ self.W2
            +
            self.b2
        )

        return self.Y_pred


    # ========================================================
    # ОБРАТНОЕ РАСПРОСТРАНЕНИЕ
    # ========================================================

    def backward(self, X, y):

        n = X.shape[0]

        # ----------------------------------------------------
        # Производная MSE
        #
        # L = mean((prediction - y)^2)
        #
        # dL/dY = 2(prediction-y)/N
        # ----------------------------------------------------

        dY = (
            2 *
            (self.Y_pred - y)
            /
            n
        )

        # ----------------------------------------------------
        # Выходной слой
        # ----------------------------------------------------

        dW2 = (
            self.A1.T
            @
            dY
        )

        db2 = np.sum(
            dY,
            axis=0,
            keepdims=True
        )

        # ----------------------------------------------------
        # Передаем ошибку на скрытый слой
        # ----------------------------------------------------

        dA1 = (
            dY
            @
            self.W2.T
        )

        # Производная ReLU

        dZ1 = (
            dA1
            *
            self.relu_derivative(
                self.Z1
            )
        )

        # ----------------------------------------------------
        # Градиенты первого слоя
        # ----------------------------------------------------

        dW1 = (
            X.T
            @
            dZ1
        )

        db1 = np.sum(
            dZ1,
            axis=0,
            keepdims=True
        )

        # ----------------------------------------------------
        # ГРАДИЕНТНЫЙ СПУСК
        # ----------------------------------------------------

        self.W2 -= (
            self.learning_rate
            *
            dW2
        )

        self.b2 -= (
            self.learning_rate
            *
            db2
        )

        self.W1 -= (
            self.learning_rate
            *
            dW1
        )

        self.b1 -= (
            self.learning_rate
            *
            db1
        )


    # ========================================================
    # ОБУЧЕНИЕ
    # ========================================================

    def train(
        self,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=500,
        batch_size=128,
        verbose=True
    ):

        rng = np.random.default_rng(
            RANDOM_SEED
        )

        for epoch in range(epochs):

            # ------------------------------------------------
            # Перемешиваем только обучающие окна.
            #
            # Временной ряд при разделении train / val / test
            # НЕ перемешивался.
            # ------------------------------------------------

            indexes = rng.permutation(
                len(X_train)
            )

            # ------------------------------------------------
            # Mini-batch gradient descent
            # ------------------------------------------------

            for start in range(
                0,
                len(X_train),
                batch_size
            ):

                batch_indexes = indexes[
                    start:start + batch_size
                ]

                X_batch = X_train[
                    batch_indexes
                ]

                y_batch = y_train[
                    batch_indexes
                ]

                # Прямой проход

                self.forward(
                    X_batch
                )

                # Обратный проход

                self.backward(
                    X_batch,
                    y_batch
                )

            # ------------------------------------------------
            # Ошибка после эпохи
            # ------------------------------------------------

            train_predictions = self.forward(
                X_train
            )

            train_loss = self.mse(
                y_train,
                train_predictions
            )

            val_predictions = self.forward(
                X_val
            )

            val_loss = self.mse(
                y_val,
                val_predictions
            )

            self.history[
                "train_loss"
            ].append(train_loss)

            self.history[
                "val_loss"
            ].append(val_loss)

            # ------------------------------------------------
            # Вывод информации
            # ------------------------------------------------

            if verbose:

                if (
                    epoch == 0
                    or
                    (epoch + 1) % 50 == 0
                ):

                    print(
                        f"Эпоха "
                        f"{epoch + 1:4d}/{epochs} | "
                        f"Train MSE: "
                        f"{train_loss:.6f} | "
                        f"Validation MSE: "
                        f"{val_loss:.6f}"
                    )


    # ========================================================
    # ПРОГНОЗ
    # ========================================================

    def predict(self, X):

        return self.forward(X)


# ============================================================
# 9. МЕТРИКИ
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    mae = np.mean(
        np.abs(
            y_true - y_pred
        )
    )

    rmse = np.sqrt(
        np.mean(
            (y_true - y_pred) ** 2
        )
    )

    return mae, rmse


# ============================================================
# 10. ГРАФИК ОБУЧЕНИЯ
# ============================================================

def plot_training_history(model):

    plt.figure(figsize=(10, 5))

    plt.plot(
        model.history["train_loss"],
        label="Train"
    )

    plt.plot(
        model.history["val_loss"],
        label="Validation"
    )

    plt.xlabel("Эпоха")
    plt.ylabel("MSE")

    plt.title(
        "Изменение ошибки во время обучения"
    )

    plt.legend()

    plt.grid()

    plt.tight_layout()

    plt.show()


# ============================================================
# 11. ГРАФИК ПРОГНОЗА
# ============================================================

def plot_predictions(
    timestamps,
    real,
    predicted,
    count=300
):

    timestamps = timestamps[-count:]

    real = real[-count:]

    predicted = predicted[-count:]

    plt.figure(
        figsize=(14, 6)
    )

    plt.plot(
        timestamps,
        real,
        label="Фактические значения"
    )

    plt.plot(
        timestamps,
        predicted,
        label="Прогноз нейронной сети"
    )

    plt.xlabel("Время")

    plt.ylabel(
        "Количество велосипедов"
    )

    plt.title(
        "Фактические значения и прогноз"
    )

    plt.legend()

    plt.grid()

    plt.tight_layout()

    plt.show()


# ============================================================
# 12. ОСНОВНАЯ ПРОГРАММА
# ============================================================

def main():

    # ========================================================
    # ШАГ 1. Получаем данные
    # ========================================================

    csv_path = download_dataset()

    df = load_dataframe(
        csv_path
    )

    print("\nПервые строки данных:")

    print(
        df.head()
    )

    # ========================================================
    # ШАГ 2. Получаем временной ряд
    # ========================================================

    values, timestamps = prepare_series(
        df
    )

    print(
        "\nКоличество наблюдений:",
        len(values)
    )

    print(
        "Минимум:",
        values.min()
    )

    print(
        "Максимум:",
        values.max()
    )

    print(
        "Среднее:",
        values.mean()
    )

    # Исходный ряд

    plot_original_series(
        values,
        timestamps
    )

    # ========================================================
    # ШАГ 3. Разделение по времени
    # ========================================================

    n = len(values)

    train_end = int(
        n * TRAIN_PART
    )

    val_end = int(
        n * (
            TRAIN_PART
            +
            VAL_PART
        )
    )

    print("\nРазделение данных:")

    print(
        "Train:",
        train_end
    )

    print(
        "Validation:",
        val_end - train_end
    )

    print(
        "Test:",
        n - val_end
    )

    # ========================================================
    # ВАЖНО
    #
    # minimum и maximum вычисляются
    # ТОЛЬКО по Train
    # ========================================================

    train_values = values[
        :train_end
    ]

    train_min = train_values.min()

    train_max = train_values.max()

    print(
        "\nTrain min:",
        train_min
    )

    print(
        "Train max:",
        train_max
    )

    # ========================================================
    # ШАГ 4. Нормализация
    # ========================================================

    normalized_values = normalize(
        values,
        train_min,
        train_max
    )

    # ========================================================
    # ШАГ 5. Создаем окна
    # ========================================================

    # -------------------------------
    # Train
    # -------------------------------

    X_train, y_train = create_windows(
        normalized_values[
            :train_end
        ],
        WINDOW_SIZE
    )

    # -------------------------------
    # Validation
    #
    # Добавляем предыдущие 24 часа,
    # чтобы можно было прогнозировать
    # первое значение validation.
    # -------------------------------

    val_data = normalized_values[
        train_end - WINDOW_SIZE:
        val_end
    ]

    X_val, y_val = create_windows(
        val_data,
        WINDOW_SIZE
    )

    # -------------------------------
    # Test
    # -------------------------------

    test_data = normalized_values[
        val_end - WINDOW_SIZE:
    ]

    X_test, y_test = create_windows(
        test_data,
        WINDOW_SIZE
    )

    # Временные метки для тестовых
    # целевых значений

    test_timestamps = timestamps[
        val_end:
    ]

    print("\nРазмерности:")

    print(
        "X_train:",
        X_train.shape
    )

    print(
        "y_train:",
        y_train.shape
    )

    print(
        "X_val:",
        X_val.shape
    )

    print(
        "X_test:",
        X_test.shape
    )

    # ========================================================
    # ШАГ 6.
    # Эксперимент с количеством нейронов
    # ========================================================

    models = {}

    results = []

    for hidden_size in HIDDEN_SIZES:

        print(
            "\n"
            +
            "=" * 60
        )

        print(
            f"Сеть: "
            f"{WINDOW_SIZE} -> "
            f"{hidden_size} -> 1"
        )

        print(
            "=" * 60
        )

        model = NeuralNetwork(
            input_size=WINDOW_SIZE,
            hidden_size=hidden_size,
            learning_rate=LEARNING_RATE,
            seed=RANDOM_SEED
        )

        model.train(
            X_train,
            y_train,
            X_val,
            y_val,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=True
        )

        models[
            hidden_size
        ] = model

        # ====================================================
        # Validation
        # ====================================================

        val_prediction_norm = model.predict(
            X_val
        )

        val_prediction = denormalize(
            val_prediction_norm,
            train_min,
            train_max
        )

        val_real = denormalize(
            y_val,
            train_min,
            train_max
        )

        val_mae, val_rmse = (
            calculate_metrics(
                val_real,
                val_prediction
            )
        )

        # ====================================================
        # Test
        # ====================================================

        test_prediction_norm = model.predict(
            X_test
        )

        test_prediction = denormalize(
            test_prediction_norm,
            train_min,
            train_max
        )

        test_real = denormalize(
            y_test,
            train_min,
            train_max
        )

        test_mae, test_rmse = (
            calculate_metrics(
                test_real,
                test_prediction
            )
        )

        results.append(
            {
                "Hidden neurons":
                    hidden_size,

                "Validation MAE":
                    val_mae,

                "Validation RMSE":
                    val_rmse,

                "Test MAE":
                    test_mae,

                "Test RMSE":
                    test_rmse
            }
        )

    # ========================================================
    # ШАГ 7.
    # Таблица результатов
    # ========================================================

    results_df = pd.DataFrame(
        results
    )

    print(
        "\nРЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТА"
    )

    print(
        results_df.round(2)
    )

    # ========================================================
    # Выбираем сеть по VALIDATION RMSE,
    # а не по тестовой выборке
    # ========================================================

    best_row = results_df.loc[
        results_df[
            "Validation RMSE"
        ].idxmin()
    ]

    best_hidden_size = int(
        best_row[
            "Hidden neurons"
        ]
    )

    best_model = models[
        best_hidden_size
    ]

    print(
        "\nЛучшая архитектура:"
    )

    print(
        f"{WINDOW_SIZE} -> "
        f"{best_hidden_size} -> 1"
    )

    # ========================================================
    # ШАГ 8.
    # Финальный прогноз
    # ========================================================

    prediction_norm = best_model.predict(
        X_test
    )

    prediction = denormalize(
        prediction_norm,
        train_min,
        train_max
    )

    real = denormalize(
        y_test,
        train_min,
        train_max
    )

    # ========================================================
    # ШАГ 9.
    # Метрики нейронной сети
    # ========================================================

    nn_mae, nn_rmse = (
        calculate_metrics(
            real,
            prediction
        )
    )

    print(
        "\nНейронная сеть:"
    )

    print(
        f"MAE  = {nn_mae:.2f}"
    )

    print(
        f"RMSE = {nn_rmse:.2f}"
    )

    # ========================================================
    # ШАГ 10.
    # Наивный прогноз
    #
    # y(t) = y(t-1)
    # ========================================================

    naive_norm = (
        X_test[:, -1]
        .reshape(-1, 1)
    )

    naive_prediction = denormalize(
        naive_norm,
        train_min,
        train_max
    )

    naive_mae, naive_rmse = (
        calculate_metrics(
            real,
            naive_prediction
        )
    )

    print(
        "\nНаивный прогноз "
        "y(t) = y(t-1):"
    )

    print(
        f"MAE  = {naive_mae:.2f}"
    )

    print(
        f"RMSE = {naive_rmse:.2f}"
    )

    # ========================================================
    # ШАГ 11.
    # Сравнение
    # ========================================================

    print(
        "\nСравнение:"
    )

    if nn_rmse < naive_rmse:

        improvement = (
            (
                naive_rmse
                -
                nn_rmse
            )
            /
            naive_rmse
            *
            100
        )

        print(
            "Нейронная сеть лучше "
            "наивного прогноза."
        )

        print(
            f"Улучшение RMSE: "
            f"{improvement:.2f}%"
        )

    else:

        print(
            "Нейронная сеть НЕ смогла "
            "превзойти наивный прогноз."
        )

    # ========================================================
    # ШАГ 12.
    # График обучения
    # ========================================================

    plot_training_history(
        best_model
    )

    # ========================================================
    # ШАГ 13.
    # График прогноза
    # ========================================================

    plot_predictions(
        test_timestamps,
        real.flatten(),
        prediction.flatten(),
        count=300
    )


# ============================================================
# ЗАПУСК
# ============================================================

if __name__ == "__main__":
    main()

#С библиотеками sklearn и tensorflow

In [ ]:
import os
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

DATA_URL = (
    "https://archive.ics.uci.edu/static/public/560/"
    "seoul%2Bbike%2Bsharing%2Bdemand.zip"
)

DATA_DIR = "data"
ZIP_PATH = os.path.join(DATA_DIR, "seoul_bike.zip")

WINDOW_SIZE = 24

TRAIN_PART = 0.70
VAL_PART = 0.15

HIDDEN_SIZES = [4, 8, 16]

EPOCHS = 200
BATCH_SIZE = 32
LEARNING_RATE = 0.001

RANDOM_SEED = 42


# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================================

def download_dataset():

    os.makedirs(DATA_DIR, exist_ok=True)

    csv_path = os.path.join(
        DATA_DIR,
        "SeoulBikeData.csv"
    )

    if os.path.exists(csv_path):

        print("Данные уже загружены.")

        return csv_path

    print("Скачивание данных...")

    urllib.request.urlretrieve(
        DATA_URL,
        ZIP_PATH
    )

    with zipfile.ZipFile(
        ZIP_PATH,
        "r"
    ) as archive:

        archive.extractall(
            DATA_DIR
        )

    print("Данные загружены.")

    return csv_path


# ============================================================
# 3. ЧТЕНИЕ CSV
# ============================================================

def load_dataframe(path):

    encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]

    for encoding in encodings:

        try:

            df = pd.read_csv(
                path,
                encoding=encoding
            )

            print(
                f"Кодировка: {encoding}"
            )

            return df

        except UnicodeDecodeError:

            pass

    raise RuntimeError(
        "Не удалось прочитать файл"
    )


# ============================================================
# 4. ПОДГОТОВКА РЯДА
# ============================================================

def prepare_series(df):

    target = "Rented Bike Count"

    # Создаем полноценное время:
    # дата + час

    df["Timestamp"] = (
        pd.to_datetime(
            df["Date"],
            dayfirst=True
        )
        +
        pd.to_timedelta(
            df["Hour"],
            unit="h"
        )
    )

    df = df.sort_values(
        "Timestamp"
    )

    values = (
        df[target]
        .astype(float)
        .values
        .reshape(-1, 1)
    )

    timestamps = (
        df["Timestamp"]
        .values
    )

    return values, timestamps


# ============================================================
# 5. СОЗДАНИЕ ВРЕМЕННЫХ ОКОН
# ============================================================

def create_windows(
    data,
    window_size
):

    X = []
    y = []

    for i in range(
        window_size,
        len(data)
    ):

        X.append(
            data[
                i - window_size:i,
                0
            ]
        )

        y.append(
            data[i, 0]
        )

    return (
        np.array(X),
        np.array(y)
    )


# ============================================================
# 6. СОЗДАНИЕ НЕЙРОННОЙ СЕТИ
# ============================================================

def create_model(
    input_size,
    hidden_size
):

    model = Sequential(

        [
            Input(
                shape=(input_size,)
            ),

            Dense(
                hidden_size,
                activation="relu"
            ),

            Dense(
                1
            )
        ]

    )

    optimizer = Adam(
        learning_rate=LEARNING_RATE
    )

    model.compile(

        optimizer=optimizer,

        loss="mse",

        metrics=["mae"]

    )

    return model


# ============================================================
# 7. МЕТРИКИ
# ============================================================

def calculate_metrics(
    real,
    predicted
):

    mae = mean_absolute_error(
        real,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            real,
            predicted
        )
    )

    return mae, rmse


# ============================================================
# 8. ГРАФИК ОБУЧЕНИЯ
# ============================================================

def plot_history(history):

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        history.history["loss"],
        label="Train"
    )

    plt.plot(
        history.history["val_loss"],
        label="Validation"
    )

    plt.xlabel(
        "Эпоха"
    )

    plt.ylabel(
        "MSE"
    )

    plt.title(
        "Изменение ошибки"
    )

    plt.legend()

    plt.grid()

    plt.show()


# ============================================================
# 9. ГРАФИК ПРОГНОЗА
# ============================================================

def plot_predictions(
    timestamps,
    real,
    prediction,
    count=300
):

    plt.figure(
        figsize=(14, 6)
    )

    plt.plot(

        timestamps[-count:],

        real[-count:],

        label="Фактические значения"
    )

    plt.plot(

        timestamps[-count:],

        prediction[-count:],

        label="Прогноз"
    )

    plt.xlabel(
        "Время"
    )

    plt.ylabel(
        "Количество велосипедов"
    )

    plt.title(
        "Прогноз нейронной сети"
    )

    plt.legend()

    plt.grid()

    plt.show()


# ============================================================
# 10. ОСНОВНАЯ ПРОГРАММА
# ============================================================

def main():

    # --------------------------------------------------------
    # Загружаем данные
    # --------------------------------------------------------

    path = download_dataset()

    df = load_dataframe(
        path
    )

    print(
        "\nПервые строки:"
    )

    print(
        df.head()
    )

    # --------------------------------------------------------
    # Получаем временной ряд
    # --------------------------------------------------------

    values, timestamps = (
        prepare_series(df)
    )

    print(
        "\nКоличество наблюдений:",
        len(values)
    )

    # --------------------------------------------------------
    # Показываем исходный ряд
    # --------------------------------------------------------

    plt.figure(
        figsize=(14, 5)
    )

    plt.plot(
        timestamps,
        values
    )

    plt.title(
        "Исходный временной ряд"
    )

    plt.xlabel(
        "Дата"
    )

    plt.ylabel(
        "Количество велосипедов"
    )

    plt.grid()

    plt.show()

    # ========================================================
    # 11. TRAIN / VALIDATION / TEST
    # ========================================================

    n = len(values)

    train_end = int(
        n * TRAIN_PART
    )

    val_end = int(
        n * (
            TRAIN_PART
            +
            VAL_PART
        )
    )

    print(
        "\nTrain:",
        train_end
    )

    print(
        "Validation:",
        val_end - train_end
    )

    print(
        "Test:",
        n - val_end
    )

    # ========================================================
    # 12. НОРМАЛИЗАЦИЯ
    # ========================================================

    scaler = MinMaxScaler()

    # ВАЖНО:
    # обучаем scaler только
    # на обучающей выборке

    scaler.fit(
        values[:train_end]
    )

    normalized_values = (
        scaler.transform(values)
    )

    # ========================================================
    # 13. СОЗДАНИЕ ОКОН
    # ========================================================

    # ---------------------------
    # Train
    # ---------------------------

    X_train, y_train = (
        create_windows(
            normalized_values[
                :train_end
            ],
            WINDOW_SIZE
        )
    )

    # ---------------------------
    # Validation
    # ---------------------------

    val_data = (
        normalized_values[
            train_end
            -
            WINDOW_SIZE
            :
            val_end
        ]
    )

    X_val, y_val = (
        create_windows(
            val_data,
            WINDOW_SIZE
        )
    )

    # ---------------------------
    # Test
    # ---------------------------

    test_data = (
        normalized_values[
            val_end
            -
            WINDOW_SIZE
            :
        ]
    )

    X_test, y_test = (
        create_windows(
            test_data,
            WINDOW_SIZE
        )
    )

    test_timestamps = (
        timestamps[val_end:]
    )

    print(
        "\nРазмер X_train:",
        X_train.shape
    )

    print(
        "Размер X_val:",
        X_val.shape
    )

    print(
        "Размер X_test:",
        X_test.shape
    )

    # ========================================================
    # 14. ОБУЧЕНИЕ РАЗНЫХ СЕТЕЙ
    # ========================================================

    models = {}

    histories = {}

    results = []

    for hidden_size in HIDDEN_SIZES:

        print(
            "\n"
            +
            "=" * 60
        )

        print(
            f"Модель "
            f"{WINDOW_SIZE} -> "
            f"{hidden_size} -> 1"
        )

        print(
            "=" * 60
        )

        model = create_model(
            WINDOW_SIZE,
            hidden_size
        )

        model.summary()

        # ----------------------------------------------------
        # EarlyStopping
        #
        # Если validation loss перестает уменьшаться,
        # обучение автоматически прекращается.
        # ----------------------------------------------------

        early_stopping = EarlyStopping(

            monitor="val_loss",

            patience=15,

            restore_best_weights=True
        )

        history = model.fit(

            X_train,
            y_train,

            validation_data=(
                X_val,
                y_val
            ),

            epochs=EPOCHS,

            batch_size=BATCH_SIZE,

            callbacks=[
                early_stopping
            ],

            verbose=0
        )

        models[
            hidden_size
        ] = model

        histories[
            hidden_size
        ] = history

        # ====================================================
        # Validation
        # ====================================================

        val_pred_norm = model.predict(
            X_val,
            verbose=0
        )

        val_pred = (
            scaler.inverse_transform(
                val_pred_norm
            )
        )

        val_real = (
            scaler.inverse_transform(
                y_val.reshape(-1, 1)
            )
        )

        val_mae, val_rmse = (
            calculate_metrics(
                val_real,
                val_pred
            )
        )

        # ====================================================
        # Test
        # ====================================================

        test_pred_norm = model.predict(
            X_test,
            verbose=0
        )

        test_pred = (
            scaler.inverse_transform(
                test_pred_norm
            )
        )

        test_real = (
            scaler.inverse_transform(
                y_test.reshape(-1, 1)
            )
        )

        test_mae, test_rmse = (
            calculate_metrics(
                test_real,
                test_pred
            )
        )

        results.append(

            {
                "Hidden neurons":
                    hidden_size,

                "Validation MAE":
                    val_mae,

                "Validation RMSE":
                    val_rmse,

                "Test MAE":
                    test_mae,

                "Test RMSE":
                    test_rmse,

                "Epochs":
                    len(
                        history.history[
                            "loss"
                        ]
                    )
            }

        )

    # ========================================================
    # 15. РЕЗУЛЬТАТЫ
    # ========================================================

    results_df = pd.DataFrame(
        results
    )

    print(
        "\nРЕЗУЛЬТАТЫ:"
    )

    print(
        results_df.round(2)
    )

    # ========================================================
    # 16. ЛУЧШАЯ МОДЕЛЬ
    # ========================================================

    best_row = results_df.loc[
        results_df[
            "Validation RMSE"
        ].idxmin()
    ]

    best_hidden_size = int(
        best_row[
            "Hidden neurons"
        ]
    )

    best_model = models[
        best_hidden_size
    ]

    best_history = histories[
        best_hidden_size
    ]

    print(
        "\nЛучшая модель:"
    )

    print(

        f"{WINDOW_SIZE} -> "
        f"{best_hidden_size} -> 1"

    )

    # ========================================================
    # 17. ФИНАЛЬНЫЙ ПРОГНОЗ
    # ========================================================

    prediction_norm = (
        best_model.predict(
            X_test,
            verbose=0
        )
    )

    prediction = (
        scaler.inverse_transform(
            prediction_norm
        )
    )

    real = (
        scaler.inverse_transform(
            y_test.reshape(-1, 1)
        )
    )

    # ========================================================
    # 18. MAE И RMSE
    # ========================================================

    nn_mae, nn_rmse = (
        calculate_metrics(
            real,
            prediction
        )
    )

    print(
        "\nНейронная сеть:"
    )

    print(
        f"MAE  = {nn_mae:.2f}"
    )

    print(
        f"RMSE = {nn_rmse:.2f}"
    )

    # ========================================================
    # 19. НАИВНЫЙ ПРОГНОЗ
    #
    # Следующее значение равно предыдущему
    # ========================================================

    naive_norm = (
        X_test[:, -1]
        .reshape(-1, 1)
    )

    naive_prediction = (
        scaler.inverse_transform(
            naive_norm
        )
    )

    naive_mae, naive_rmse = (
        calculate_metrics(
            real,
            naive_prediction
        )
    )

    print(
        "\nНаивный прогноз:"
    )

    print(
        f"MAE  = {naive_mae:.2f}"
    )

    print(
        f"RMSE = {naive_rmse:.2f}"
    )

    # ========================================================
    # 20. СРАВНЕНИЕ
    # ========================================================

    if nn_rmse < naive_rmse:

        improvement = (

            (
                naive_rmse
                -
                nn_rmse
            )

            /
            naive_rmse

            *
            100
        )

        print(
            "\nНейронная сеть лучше."
        )

        print(
            f"Улучшение RMSE: "
            f"{improvement:.2f}%"
        )

    else:

        print(
            "\nНаивный прогноз оказался лучше."
        )

    # ========================================================
    # 21. ГРАФИК ОБУЧЕНИЯ
    # ========================================================

    plot_history(
        best_history
    )

    # ========================================================
    # 22. ГРАФИК ПРОГНОЗА
    # ========================================================

    plot_predictions(

        test_timestamps,

        real.flatten(),

        prediction.flatten(),

        count=300
    )


# ============================================================
# ЗАПУСК
# ============================================================

if __name__ == "__main__":

    main()